# Import libraries

In [1]:
import pandas as pd
import os
from pathlib import Path
import shutil
from typing import List
import json 
from tqdm import tqdm

import concurrent.futures
from typing import List, Tuple, Dict
import multiprocessing

import sys
sys.path.append('..')
from utils.audio_util import convert_mp3_to_flac, resample_audios, trim_silence_with_vad, normalize_audio_files
from utils.file_util import recursive_copy
from utils.text_util import clean_text_cv, handle_maiyamok

from transformers import Wav2Vec2FeatureExtractor, WavLMForXVector
import torch
import torchaudio
from collections import defaultdict
import gc
import numpy as np
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_distances

# Read the validated tsv

In [2]:
validated_data = pd.read_csv('../data/raw/cv-corpus-20.0-2024-12-06/th/validated.tsv', sep='\t')

/tmp/ipykernel_1147454/1241631071.py:1: DtypeWarning: Columns (9,12) have mixed types. Specify dtype option on import or set low_memory=False.
  validated_data = pd.read_csv('../data/raw/cv-corpus-20.0-2024-12-06/th/validated.tsv', sep='\t')


# Define word replacement function

In [3]:
# replace_dict = {
#     "Facebook": "เฟซบุ๊ก",
#     "softmax": "ซอฟต์แม็กซ์",
#     "Astroturf": "แอสโตรเทิร์ฟ",
#     "Burke": "เบิร์ก",
#     "whilst": "ไวล์สท์",
#     "Kenny": "เคนนี",
#     "Flickr": "ฟลิกเกอร์",
#     "Asperger": "แอสเพอร์เกอร์",
#     "Johanna": "โจแอนนา",
#     "C" : "ซี",
#     "section": "เซคชัน",
#     "Mr Lincoln": "มิสเตอร์ ลินคอล์น",
#     "Brexiteers": "เบร็กซิทเทียร์ส",
#     "Brexit": "เบร็กซิท"
# }

replacing_word = [
    ['เพฃร', 'เพชร'],
    # ['เวิล์ด', 'เวิลด์'],
    # ['อัลกอรึธึม', 'อัลกอริทึม'],
    # ['เซนทรัล', 'เซ็นทรัล'],
    # ['เวิร์ล', 'เวิลด์'],
    # ['แรคคูน', 'แร็คคูน'],
    # ['เลกฮอร์น', 'เล็กฮอร์น'],
    # ['แอร์โรว์', 'แอร์โร่ว์'],
    # ['ดีเวลลอปเมนท์', 'ดีเวลลอปเม้นท์'],
    # ['เวดเดล', 'เว็ดเดล'],
    # ['ธัญาโภชน์', 'ธัญโภชน์'],
    # ['เคเบิล', 'เคเบิ้ล'],
    # ['เบตสัน', 'เบ๊ตสัน'],
    # ['นโบาย', 'นโยบาย'],
    # ['รัชาดาภิเษก', 'รัชดาภิเษก'],
    # ['่หมวกกันน็อค', 'หมวกกันน็อค'],
    # ['คาแรคเตอร์', 'คาแร็คเตอร์'],
    # ['คาแรกเตอร์', 'คาแร็คเตอร์'],
    # ['คาร์แร็คเตอร์', 'คาแร็คเตอร์']
]

# maiyamok_exceptions = {
#     'เธอเป็นคนดีมาก ๆ': 'เธอเป็นคนดีมากมาก',
#     'ให้ประสานงานกับสมาชิกคนอื่น ๆ และหารือเกี่ยวกับปัญหานี้ในภายหลัง': 'ให้ประสานงานกับสมาชิกคนอื่นคนอื่น และหารือเกี่ยวกับปัญหานี้ในภายหลัง',
#     'ช่างเป็นสวนหลังบ้านที่ดีงามมาก ๆ จริง ๆ คุณนาย !': 'ช่างเป็นสวนหลังบ้านที่ดีงามมากมาก จริงจริง คุณนาย !',
#     'ฉันอาจจะผ่านเรื่องเล็ก ๆ น้อย ๆ นั้นไปได้ด้วยดี': 'ฉันอาจจะผ่านเรื่องเล็กเล็ก น้อยน้อย นั้นไปได้ด้วยดี',
#     'มันทำให้รู้สึกสดชื่นจริง ๆ ที่ได้นั่งท่ามกลางบรรยากาศที่มีแต่ลมพัดเย็น ๆ ที่แสนสบาย': 'มันทำให้รู้สึกสดชื่นจริงจริง ที่ได้นั่งท่ามกลางบรรยากาศที่มีแต่ลมพัดเย็นเย็น ที่แสนสบาย',
#     'ซาลกำลังทำให้คนอื่น ๆ โกรธ': 'ซาลกำลังทำให้คนอื่นอื่น โกรธ',
#     'มีทันตแพทย์และแพทย์คนอื่น ๆ ในคลินิกนี้': 'มีทันตแพทย์และแพทย์คนอื่นอื่น ในคลินิกนี้',
#     'ใครพูดรัว ๆ ติดกันได้นาน ๆ บอกเราด้วย': 'ใครพูดรัวรัว ติดกันได้นานนาน บอกเราด้วย',
#     '“ผมเสียใจจริง ๆ ที่ต้องไปครับท่าน” ฉันตอบกลับ': '“ผมเสียใจจริงจริง ที่ต้องไปครับท่าน” ฉันตอบกลับ',
#     'ใจเย็น ๆ ค่ะ ไม่ต้องตื่นเต้น': 'ใจเย็นเย็น ค่ะ ไม่ต้องตื่นเต้น',
#     'เรื่องนี้มันดีมาก ๆ': 'เรื่องนี้มันดีมากมาก',
#     'คุณผู้หญิงเหมือนเด็กเล็ก ๆ': 'คุณผู้หญิงเหมือนเด็กเล็กเล็ก',
#     "'ช่างเป็นดอกไม้ดอกใหญ่ ๆ สมอย่างที่ต้องเป็นจริง ๆ !' เธอแสดงความคิดต่อมาของเธอ": "'ช่างเป็นดอกไม้ดอกใหญ่ใหญ่ สมอย่างที่ต้องเป็นจริงจริง !' เธอแสดงความคิดต่อมาของเธอ",
#     'คนอื่น ๆ เดินตามรอยของเขา' :'คนอื่นอื่น เดินตามรอยของเขา'
# }

def preprocess_words(dataframe: pd.DataFrame, column_name: str, replacing_pairs: List[List[str]]) -> pd.DataFrame:
    # dataframe[column_name] = dataframe[column_name].apply(lambda x: handle_maiyamok(x, maiyamok_exceptions))
    for old_word, new_word in replacing_pairs:
        dataframe[column_name] = dataframe[column_name].apply(lambda x: x.replace(old_word, new_word))
    # dataframe[column_name] = dataframe[column_name].apply(clean_text_cv, replace_dict=replace_dict)
    return dataframe

# Filter and group client_id that have over 100 records

In [4]:
filtered_data = validated_data[
    validated_data['client_id'].map(
        validated_data['client_id'].value_counts() >= 100
    )
]
filtered_data = preprocess_words(filtered_data, 'sentence', replacing_word)
grouped = filtered_data.groupby('client_id').agg(list)

/tmp/ipykernel_1147454/1935163709.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe[column_name] = dataframe[column_name].apply(lambda x: x.replace(old_word, new_word))


# Define new id instead of client_id

In [5]:
id_mapper = {id_: f'cv{str(i+1).zfill(3)}' for i, id_ in enumerate(filtered_data['client_id'].unique())}

grouped_data = {
    id_mapper[client_id]: list(zip(sentences, paths))
    for (client_id, (sentences, paths)) in grouped[['sentence', 'path']].iterrows()
}

# Moving files to the new folder

In [6]:
# Define paths
DEST_DIR = "../data/converted/commonvoice-to-virtual-vctk"
DEST_TEXT_PATH = os.path.join(DEST_DIR, "txt")
DEST_AUDIO_PATH = os.path.join(DEST_DIR, "wav32")
SRC_AUDIO_PATH = "../data/raw/cv-corpus-20.0-2024-12-06/th/clips"

# Number of worker threads (adjust based on your CPU)
NUM_WORKERS = multiprocessing.cpu_count()

def process_client_data(client_data: Tuple[str, List]) -> Dict:
    """
    Process all data for a single client
    
    Args:
        client_data: Tuple of (client_id, data)
    
    Returns:
        Dict with processing results
    """
    client_id, data = client_data
    results = {
        'client_id': client_id,
        'processed': 0,
        'failed': 0,
        'chars': set()
    }
    
    # Create client directories
    client_text_dir = os.path.join(DEST_TEXT_PATH, client_id)
    client_audio_dir = os.path.join(DEST_AUDIO_PATH, client_id)
    os.makedirs(client_text_dir, exist_ok=True)
    os.makedirs(client_audio_dir, exist_ok=True)
    
    for i, d in enumerate(data):
        # Write text file
        text_path = os.path.join(client_text_dir, f"{client_id}_{(i + 1):03d}.txt")
        with open(text_path, 'w') as f:
            f.write(d[0])
            results['chars'].update(d[0])
        
        # Convert audio file
        src_audio_path = os.path.join(SRC_AUDIO_PATH, d[1])
        dst_audio_path = os.path.join(
            client_audio_dir,
            f"{client_id}_{(i + 1):03d}_mic1.flac"
        )
        
        if convert_mp3_to_flac(src_audio_path, dst_audio_path):
            results['processed'] += 1
        else:
            results['failed'] += 1
    
    return results

# Clean and create directories
if os.path.exists(DEST_DIR):
    print("Clearing destination folder")
    shutil.rmtree(DEST_DIR)
os.makedirs(DEST_TEXT_PATH, exist_ok=True)
os.makedirs(DEST_AUDIO_PATH, exist_ok=True)

print(f"Starting parallel processing with {NUM_WORKERS} workers")

# Create progress bar for overall processing
with tqdm(total=len(grouped_data), desc="Processing clients") as pbar:
    all_chars = set()
    total_processed = 0
    total_failed = 0
    
    # Process clients in parallel
    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        # Submit all client processing tasks
        future_to_client = {
            executor.submit(process_client_data, (client_id, data)): client_id 
            for client_id, data in grouped_data.items()
        }
        
        # Process completed tasks
        for future in concurrent.futures.as_completed(future_to_client):
            client_id = future_to_client[future]
            try:
                result = future.result()
                all_chars.update(result['chars'])
                total_processed += result['processed']
                total_failed += result['failed']
            except Exception as e:
                print(f"Client {client_id} generated an exception: {str(e)}")
                total_failed += len(grouped_data[client_id])
            pbar.update(1)

print("\nConversion Summary:")
print(f"Total files processed successfully: {total_processed}")
print(f"Total files failed: {total_failed}")
print(f"Total unique characters: {len(all_chars)}")
print("Restructuring and conversion complete")

Starting parallel processing with 16 workers


Processing clients: 100%|██████████| 134/134 [1:01:26<00:00, 27.51s/it] 


Conversion Summary:
Total files processed successfully: 92956
Total files failed: 0
Total unique characters: 115
Restructuring and conversion complete


# Resample and trim audio

In [7]:
# Create destination directory if it doesn't exist
os.makedirs("../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed", exist_ok=True)

# Copy all files from wav32 to wav16_silence_trimmed
src_dir = "../data/converted/commonvoice-to-virtual-vctk/wav32"
dst_dir = "../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed"

recursive_copy(src_dir, dst_dir)

In [8]:
# Resample all files in wav16_silence_trimmed to 16kHz
SAMPLE_RATE = 16000
NUM_RESAMPLE_THREADS = 4

resample_audios(
  input_folders=dst_dir,
  file_ext="flac",
  sample_rate=SAMPLE_RATE,
  n_jobs=NUM_RESAMPLE_THREADS
)

Resampling the audio files...
Found 92956 files...


100%|██████████| 92956/92956 [01:36<00:00, 965.11it/s] 


Done !


In [9]:
# Trim silence at the beginning and end of each audio file
trim_silence_with_vad(
  input_folder=dst_dir,
  file_extension="flac",
)

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/ming/.cache/torch/hub/master.zip


Found 92956 .flac files to process


Processing files:   5%|▍         | 4586/92956 [04:37<1:13:11, 20.12it/s]

> The file ../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed/cv131/cv131_2196_mic1.flac probably does not have speech please check it !!


Processing files:  11%|█▏        | 10466/92956 [10:39<1:10:58, 19.37it/s]

...

> The file ../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed/cv132/cv132_116_mic1.flac probably does not have speech please check it !!


Processing files:  96%|█████████▌| 89091/92956 [1:32:11<02:56, 21.93it/s]

> The file ../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed/cv132/cv132_4774_mic1.flac probably does not have speech please check it !!


Processing files: 100%|██████████| 92956/92956 [1:35:54<00:00, 16.15it/s]


Processing complete

Found 29 files with no speech. List saved to ../data/converted/commonvoice-to-virtual-vctk/no_speech_files.txt


In [10]:
# Normalize the volume of all audio files to -27dB
normalize_audio_files(
  input_dir=dst_dir,
  extensions="flac",
  target_db=-27,
  sample_rate=16000,
)

Normalizing audio files: 100%|██████████| 92956/92956 [5:41:00<00:00,  4.54it/s]  


# New Grouping for virtual speakerId

In [11]:
# Clean and create directories
if os.path.exists(DEST_AUDIO_PATH):
    print("Clearing destination folder")
    shutil.rmtree(DEST_AUDIO_PATH)

Clearing destination folder


In [34]:
# Generate DVector using wavlm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('microsoft/wavlm-base-plus-sv')
model = WavLMForXVector.from_pretrained('microsoft/wavlm-base-plus-sv').to(device)

if not os.path.exists('../data/converted/commonvoice-to-virtual-vctk/dvectors.pth'):
    speaker_mapping = {}
    for speaker in tqdm(os.listdir(dst_dir)):
        for audio_file in os.listdir(os.path.join(dst_dir, speaker)):
            audio_path = os.path.join(dst_dir, speaker, audio_file)
            audio_input, sr = torchaudio.load(audio_path)
            audio_input = audio_input
            audio_array = audio_input.detach().cpu().numpy()
            with torch.no_grad():
                inputs = feature_extractor(audio_array, sampling_rate=16000, return_tensors="pt")
                inputs = {k: v.to(device) for k, v in inputs.items()}
                embeddings = model(**inputs).embeddings
                embeddings = torch.nn.functional.normalize(embeddings, dim=-1).cpu()
                if embeddings.isnan().any():
                    print(f"The embedding of {audio_file} is NaN")
                    continue
            speaker_mapping[audio_file] = {}
            speaker_mapping[audio_file]['embedding'] = embeddings
            speaker_mapping[audio_file]['name'] = speaker
    torch.save(speaker_mapping, '../data/converted/commonvoice-to-virtual-vctk/dvectors.pth')
else:
    speaker_mapping = torch.load('../data/converted/commonvoice-to-virtual-vctk/dvectors.pth')

  0%|          | 0/134 [00:00<?, ?it/s]/home/ming/.cache/pypoetry/virtualenvs/speech-dataset-converter-aRtuyZwp-py3.11/lib/python3.11/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(
 39%|███▉      | 52/134 [04:08<07:09,  5.24s/it]/home/ming/.cache/pypoetry/virtualenvs/speech-dataset-converter-aRtuyZwp-py3.11/lib/python3.11/site-packages/transformers/models/wavlm/modeling_wavlm.py:1833: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  std_features.append(hidden_states[i, :length].std(dim=0))


The embedding of cv044_098_mic1.flac is NaN
The embedding of cv044_124_mic1.flac is NaN


 55%|█████▌    | 74/134 [11:10<04:53,  4.89s/it]   

The embedding of cv133_8443_mic1.flac is NaN
The embedding of cv133_3365_mic1.flac is NaN


 57%|█████▋    | 77/134 [13:24<21:40, 22.82s/it]

...

 88%|████████▊ | 118/134 [17:01<01:45,  6.62s/it]

The embedding of cv054_120_mic1.flac is NaN


 96%|█████████▌| 128/134 [17:39<00:17,  2.95s/it]

The embedding of cv132_5823_mic1.flac is NaN
The embedding of cv132_737_mic1.flac is NaN


100%|██████████| 134/134 [19:44<00:00,  8.84s/it]


In [36]:
BATCH_SIZE = 10000  # Adjust based on your system's RAM

def process_speaker(name, dvector_items, verbose=False):
    total_items = len(dvector_items)
    
    # Prepare embeddings
    dvector_embeddings = np.array([value['embedding'] for _, value in dvector_items])
    dvector_embeddings = dvector_embeddings.squeeze()
            
    cosine_distances_matrix = cosine_distances(dvector_embeddings, dvector_embeddings)

    dvector_clusterer = AgglomerativeClustering(
        metric="precomputed",
        linkage="average",
        distance_threshold=1-0.85,
        n_clusters=None
    ).fit(cosine_distances_matrix)

    labels = defaultdict(int)
    for label in dvector_clusterer.labels_:
        labels[label] += 1

    if verbose: print([(key, labels[key]) for key in sorted(labels, key=labels.get, reverse=True)])

    # if value < 10 change key to -1
    for i in range(total_items):
        if labels[dvector_clusterer.labels_[i]] < 10:
            dvector_clusterer.labels_[i] = -1

    dvector_labels = dvector_clusterer.labels_
    
    concat_labels = defaultdict(list)
    cluster_stats = defaultdict(lambda: {'file_count': 0})
    
    for i in range(total_items):
        key = dvector_items[i][0]  # Extract filename (key)
        cluster_key = f'{dvector_labels[i]}'
        concat_labels[cluster_key].append(key)
        cluster_stats[cluster_key]['file_count'] += 1

    gc.collect()

    return concat_labels

# Main processing loop
speaker_dict_dvector = defaultdict(list)

for k, v in speaker_mapping.items():
    name = v['name']
    speaker_dict_dvector[name].append((k, v))

global_speaker_dict = {}
for name in tqdm(speaker_dict_dvector, desc="Processing speakers"):
    concat_labels = process_speaker(name, speaker_dict_dvector[name])
    for cluster_key, cluster_files in concat_labels.items():
        if cluster_key == '-1':
            continue
        global_speaker_dict[f'v{name}_{cluster_key}'] = cluster_files

json.dump(global_speaker_dict, open('../data/converted/commonvoice-to-virtual-vctk/virtual_speaker_mapping.json', 'w'), indent=4)

Processing speakers: 100%|██████████| 134/134 [01:34<00:00,  1.41it/s]


In [40]:
for speaker in tqdm(global_speaker_dict, desc="Processing speakers"):
    speaker_name = speaker
    speaker_audio_files = global_speaker_dict[speaker]
    speaker_text_files = [os.path.join(DEST_TEXT_PATH, f"{audio_file.split('_')[0]}/{'_'.join(audio_file.split('.')[0].split('_')[:2])}.txt") for audio_file in speaker_audio_files]

    speaker_audio_dir = os.path.join("../data/converted/commonvoice-to-virtual-vctk/wav16_silence_trimmed", speaker_name)
    os.makedirs(speaker_audio_dir, exist_ok=True)

    for audio_file in speaker_audio_files:
        src_audio_path = os.path.join(dst_dir, audio_file.split('_')[0], audio_file)
        dst_audio_path = os.path.join(speaker_audio_dir, audio_file)
        shutil.copy(src_audio_path, dst_audio_path)

    speaker_text_dir = os.path.join("../data/converted/commonvoice-to-virtual-vctk/txt", speaker_name)
    os.makedirs(speaker_text_dir, exist_ok=True)
    
    for text_file in speaker_text_files:
        src_text_path = text_file
        dst_text_path = os.path.join(speaker_text_dir, os.path.basename(text_file))
        shutil.copy(src_text_path, dst_text_path)

Processing speakers: 100%|██████████| 361/361 [00:36<00:00,  9.95it/s]


In [42]:
# remove folders with cv prefix in dst_dir
for speaker in os.listdir(dst_dir):
    if speaker.startswith('cv'):
        shutil.rmtree(os.path.join(dst_dir, speaker))

for speaker in os.listdir("../data/converted/commonvoice-to-virtual-vctk/txt"):
    if speaker.startswith('cv'):
        shutil.rmtree(os.path.join("../data/converted/commonvoice-to-virtual-vctk/txt", speaker))